# Multimodal SLM via Embedding Distillation

**Goal:** Teach a frozen small LLM to 'see' by training only a lightweight projection adapter
that maps CLIP/SigLIP vision embeddings into the LLM's text embedding space.

**Architecture:** `[Frozen CLIP/SigLIP] → [Trainable Adapter] → [Frozen Phi-2 / Gemma-2B]`

---
## Notebook Outline
- **Phase 1** – Environment setup & model loading
- **Phase 2** – Adapter architecture overview & sanity checks
- **Phase 3** – Alignment training
- **Phase 4** – Evaluation (VQA accuracy, zero-shot captioning, embedding metrics)
- **Phase 5** – Ablation studies & interpretability


---
## Phase 1 · Foundation Setup

In [ ]:
# ── 1.1  Install dependencies ────────────────────────────────────────────────
# Run once per Colab session.
!pip install -q transformers accelerate datasets Pillow tqdm
!pip install -q pycocoevalcap pycocotools   # for BLEU / CIDEr metrics
!pip install -q scikit-learn matplotlib     # for t-SNE & plots
# Optional: wandb for experiment tracking
# !pip install -q wandb && wandb login

In [ ]:
# ── 1.2  Verify GPU ──────────────────────────────────────────────────────────
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {device}")
if device == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── 1.3  Clone / set up project repo (if not already done) ──────────────────
import sys, os

PROJECT_ROOT = "/content/multimodal_slm"   # adjust if running locally
if os.path.isdir(PROJECT_ROOT):
    sys.path.insert(0, PROJECT_ROOT)
else:
    # If you've uploaded the helper files to Colab's /content
    raise RuntimeError(f"Project root not found at {PROJECT_ROOT}. "
                       "Upload the helper .py files or clone the repo.")

In [ ]:
# ── 1.4  Set seeds for reproducibility ──────────────────────────────────────
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if device == "cuda":
    torch.cuda.manual_seed_all(SEED)

print("Seeds fixed ✓")

In [ ]:
# ── 1.5  Download COCO Captions (run ONCE, takes ~5-10 min on Colab) ─────────
# Skip this cell if you already have COCO downloaded.

COCO_ROOT = "/content/coco"
os.makedirs(COCO_ROOT, exist_ok=True)

# Annotations
!wget -q -nc http://images.cocodataset.org/annotations/annotations_trainval2017.zip -P {COCO_ROOT}
!unzip -q -n {COCO_ROOT}/annotations_trainval2017.zip -d {COCO_ROOT}

# Images (train2017 is 18 GB – use val2017 for a quick start)
# For training, uncomment the train line:
# !wget -q -nc http://images.cocodataset.org/zips/train2017.zip -P {COCO_ROOT}
# !unzip -q -n {COCO_ROOT}/train2017.zip -d {COCO_ROOT}

!wget -q -nc http://images.cocodataset.org/zips/val2017.zip   -P {COCO_ROOT}
!unzip -q -n {COCO_ROOT}/val2017.zip   -d {COCO_ROOT}

print("COCO ready ✓")

In [ ]:
# ── 1.6  Choose & load experiment config ─────────────────────────────────────
from utils.config import get_config

# Options: "baseline", "ablation_mlp", "ablation_perceiver",
#          "ablation_contrastive", "ablation_patches", "smoke_test"
cfg = get_config("smoke_test")  # ← Start here; switch to "baseline" for real training

# Override paths for Colab
cfg.coco_root  = COCO_ROOT
cfg.output_dir = "/content/checkpoints"

---
## Phase 2 · Adapter Architecture

In [ ]:
# ── 2.1  Build the VisionLanguageModel ───────────────────────────────────────
# This loads the frozen CLIP encoder + chosen adapter + frozen Phi-2 / Gemma-2B.
# First run will download model weights (~5 GB total).

from models.adapter import VisionLanguageModel

# Build adapter kwargs based on config
adapter_kwargs = {}
if cfg.adapter_type == "mlp":
    adapter_kwargs["hidden_scale"] = cfg.mlp_hidden_scale
elif cfg.adapter_type == "perceiver":
    adapter_kwargs["num_queries"] = cfg.perceiver_num_queries
    adapter_kwargs["num_heads"]   = cfg.perceiver_num_heads
    adapter_kwargs["num_layers"]  = cfg.perceiver_num_layers

model = VisionLanguageModel(
    vision_model_name = cfg.vision_model_name,
    lm_model_name     = cfg.lm_model_name,
    adapter_type      = cfg.adapter_type,
    use_cls_only      = cfg.use_cls_only,
    vision_dim        = cfg.vision_dim,
    text_dim          = cfg.text_dim,
    **adapter_kwargs,
)
model.adapter = model.adapter.to(device)

print(f"\nTrainable parameters: {model.num_trainable_params():,}")
print(f"Total LM parameters:  {sum(p.numel() for p in model.lm.parameters()):,}")

In [ ]:
# ── 2.2  Architecture summary ────────────────────────────────────────────────
print("=" * 60)
print("VISION ENCODER (frozen)")
print(model.vision_encoder)
print("\n" + "=" * 60)
print("ADAPTER (trainable)")
print(model.adapter)
print("\n" + "=" * 60)
print("LANGUAGE MODEL – first 2 layers shown (frozen)")
# Show just the embedding layer + first transformer block to avoid wall of text
print(model.lm.model.embed_tokens)
print("...")

In [ ]:
# ── 2.3  Sanity check: single forward pass ───────────────────────────────────
from transformers import AutoTokenizer
from data.dataset import get_processor
from PIL import Image
import requests
from io import BytesIO

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(cfg.lm_model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load processor
processor = get_processor(cfg.vision_model_name)

# Download a test image
url = "http://images.cocodataset.org/val2017/000000039769.jpg"  # cats on a couch
resp = requests.get(url, timeout=10)
image = Image.open(BytesIO(resp.content)).convert("RGB")
image

In [ ]:
# ── 2.4  Run a forward pass and check output shapes ──────────────────────────
pixel_values = processor(images=image, return_tensors="pt")["pixel_values"].to(device)
prompt       = "What is in this image?"
inputs       = tokenizer(prompt, return_tensors="pt")
input_ids    = inputs["input_ids"].to(device)

# Check visual embedding shape
with torch.no_grad():
    vis_emb = model.encode_images(pixel_values)

print(f"pixel_values shape  : {pixel_values.shape}")
print(f"visual embeds shape : {vis_emb.shape}")
print(f"input_ids shape     : {input_ids.shape}")
print("\nAll shapes look good! ✓")

In [ ]:
# ── 2.5  Sanity check: greedy generation (untrained model) ───────────────────
# Outputs will be incoherent – that's expected before training.

with torch.no_grad():
    generated_ids = model.generate(
        pixel_values  = pixel_values,
        input_ids     = input_ids,
        attention_mask= inputs["attention_mask"].to(device),
        max_new_tokens= 40,
        do_sample     = False,
    )

output_text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
print("[Untrained model output]")
print(output_text)

---
## Phase 3 · Alignment Training

In [ ]:
# ── 3.1  Build datasets ──────────────────────────────────────────────────────
from data.dataset import COCOCaptionsDataset, build_dataloader

train_dataset = COCOCaptionsDataset(
    coco_root   = cfg.coco_root,
    split       = "train",      # use "val" if you only downloaded val images
    tokenizer   = tokenizer,
    processor   = processor,
    max_length  = cfg.max_caption_length,
    max_samples = cfg.max_train_samples,
)

val_dataset = COCOCaptionsDataset(
    coco_root   = cfg.coco_root,
    split       = "val",
    tokenizer   = tokenizer,
    processor   = processor,
    max_length  = cfg.max_caption_length,
    max_samples = cfg.max_val_samples,
)

train_loader = build_dataloader(train_dataset, batch_size=cfg.batch_size, shuffle=True,  num_workers=cfg.num_workers)
val_loader   = build_dataloader(val_dataset,   batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers)

print(f"Train: {len(train_dataset):,} samples  ({len(train_loader)} batches)")
print(f"Val  : {len(val_dataset):,} samples  ({len(val_loader)} batches)")

In [ ]:
# ── 3.2  Inspect a training batch ────────────────────────────────────────────
batch = next(iter(train_loader))
print("Batch keys         :", list(batch.keys()))
print("pixel_values shape :", batch["pixel_values"].shape)
print("input_ids shape    :", batch["input_ids"].shape)
print("labels shape       :", batch["labels"].shape)
print("\nSample caption     :", batch["caption"][0])

In [ ]:
# ── 3.3  Measure pre-training baseline embedding alignment (CKA) ─────────────
# Run this BEFORE training to establish a baseline CKA value.
# Target after training: CKA > 0.7

from evaluation.evaluator import EmbeddingAnalyzer

analyzer = EmbeddingAnalyzer()
cka_before = analyzer.compute_cka(model, val_loader, device=device, max_batches=20)
print(f"\nPre-training CKA: {cka_before:.4f}  (baseline)")

In [ ]:
# ── 3.4  t-SNE: visualise embedding space BEFORE training ────────────────────
fig_before = analyzer.tsne_plot(
    model, val_loader, device=device, max_batches=10,
    save_path="/content/tsne_before.png"
)

In [ ]:
# ── 3.5  Initialise trainer ───────────────────────────────────────────────────
from training.trainer import Trainer

trainer = Trainer(
    model              = model,
    train_loader       = train_loader,
    val_loader         = val_loader,
    output_dir         = cfg.output_dir,
    num_epochs         = cfg.num_epochs,
    learning_rate      = cfg.learning_rate,
    weight_decay       = cfg.weight_decay,
    warmup_ratio       = cfg.warmup_ratio,
    grad_accum_steps   = cfg.grad_accum_steps,
    max_grad_norm      = cfg.max_grad_norm,
    use_contrastive    = cfg.use_contrastive,
    contrastive_weight = cfg.contrastive_weight,
    use_l2             = cfg.use_l2,
    l2_weight          = cfg.l2_weight,
    log_every          = cfg.log_every,
    save_every         = cfg.save_every,
    use_wandb          = cfg.use_wandb,
)

# To resume from a checkpoint:
# trainer.load_checkpoint("/content/checkpoints/adapter_step500.pt")

In [ ]:
# ── 3.6  TRAIN ───────────────────────────────────────────────────────────────
# Expected time on T4 (16 GB):
#   smoke_test (500 samples, 1 epoch) : ~5 min
#   baseline   (COCO, 3 epochs)       : ~8-12 hours
#   cc3m       (3.3M, 1 epoch)        : ~18-20 hours

trainer.train()

In [ ]:
# ── 3.7  Load best checkpoint for evaluation ──────────────────────────────────
import os
# Find latest checkpoint
ckpts = sorted([f for f in os.listdir(cfg.output_dir) if f.endswith(".pt")])
if ckpts:
    best_ckpt = os.path.join(cfg.output_dir, ckpts[-1])
    print(f"Loading: {best_ckpt}")
    ckpt = torch.load(best_ckpt, map_location="cpu")
    model.adapter.load_state_dict(ckpt["adapter_state_dict"])
    print("Checkpoint loaded ✓")

---
## Phase 4 · Evaluation

In [ ]:
# ── 4.1  Qualitative: generate captions for a few images ────────────────────
import requests
from io import BytesIO
import matplotlib.pyplot as plt

TEST_IMAGES = [
    ("http://images.cocodataset.org/val2017/000000039769.jpg", "Two cats on a couch"),
    ("http://images.cocodataset.org/val2017/000000397133.jpg", "Baseball game"),
    ("http://images.cocodataset.org/val2017/000000252219.jpg", "Elephant in water"),
    ("http://images.cocodataset.org/val2017/000000324158.jpg", "Pizza on a table"),
]

fig, axes = plt.subplots(1, len(TEST_IMAGES), figsize=(18, 5))
model.eval()
prompt_str = "Describe this image:"
prompt_ids = tokenizer(prompt_str, return_tensors="pt")["input_ids"].to(device)

for ax, (url, label) in zip(axes, TEST_IMAGES):
    img = Image.open(BytesIO(requests.get(url, timeout=10).content)).convert("RGB")
    pv  = processor(images=img, return_tensors="pt")["pixel_values"].to(device)

    with torch.no_grad():
        gen_ids = model.generate(
            pixel_values  = pv,
            input_ids     = prompt_ids,
            attention_mask= torch.ones_like(prompt_ids),
            max_new_tokens= 50,
            do_sample     = False,
        )
    caption = tokenizer.decode(gen_ids[0][prompt_ids.shape[-1]:], skip_special_tokens=True)

    ax.imshow(img)
    ax.axis("off")
    ax.set_title(f"GT: {label}\n\nPred: {caption[:80]}", fontsize=8, wrap=True)

plt.tight_layout()
plt.savefig("/content/qualitative_results.png", dpi=150)
plt.show()

In [ ]:
# ── 4.2  Post-training embedding alignment (CKA) ─────────────────────────────
cka_after = analyzer.compute_cka(model, val_loader, device=device, max_batches=20)
print(f"\nCKA before training : {cka_before:.4f}")
print(f"CKA after training  : {cka_after:.4f}")
print(f"Improvement         : {cka_after - cka_before:+.4f}")

In [ ]:
# ── 4.3  t-SNE AFTER training (compare to before) ────────────────────────────
fig_after = analyzer.tsne_plot(
    model, val_loader, device=device, max_batches=10,
    save_path="/content/tsne_after.png"
)

# Side-by-side comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(plt.imread("/content/tsne_before.png"))
axes[0].set_title("Before Training"); axes[0].axis("off")
axes[1].imshow(plt.imread("/content/tsne_after.png"))
axes[1].set_title("After Training"); axes[1].axis("off")
plt.tight_layout()
plt.savefig("/content/tsne_comparison.png", dpi=150)
plt.show()

In [ ]:
# ── 4.4  Cosine similarity stats ─────────────────────────────────────────────
sim_stats = analyzer.cosine_similarity_stats(model, val_loader, device=device, max_batches=50)
# Good alignment: matched_sim >> unmatched_sim

In [ ]:
# ── 4.5  VQA Evaluation ──────────────────────────────────────────────────────
# Requires VQAv2 data. Download separately:
#   wget https://s3.amazonaws.com/cvmlp/vqa/mscoco/vqa/v2_Questions_Val_mscoco.zip
#   wget https://s3.amazonaws.com/cvmlp/vqa/mscoco/vqa/v2_Annotations_Val_mscoco.zip

VQA_ROOT = "/content/vqa"  # adjust path

if os.path.isdir(VQA_ROOT):
    from data.dataset import VQAv2Dataset, build_dataloader
    from evaluation.evaluator import VQAEvaluator

    vqa_dataset = VQAv2Dataset(
        vqa_root    = VQA_ROOT,
        processor   = processor,
        tokenizer   = tokenizer,
        max_samples = 2000,   # subset for speed; full val = 214k
    )
    vqa_loader = build_dataloader(vqa_dataset, batch_size=8, shuffle=False)

    vqa_eval = VQAEvaluator(model, tokenizer, processor, device=device)
    vqa_results = vqa_eval.evaluate(vqa_loader, max_new_tokens=5)
else:
    print(f"VQA data not found at {VQA_ROOT}. Skipping VQA evaluation.")
    print("Download instructions in the cell comment above.")

In [ ]:
# ── 4.6  Zero-shot captioning (BLEU-1 approximation) ─────────────────────────
from evaluation.evaluator import CaptionEvaluator

cap_eval = CaptionEvaluator(model, tokenizer, processor, device=device)
cap_records = cap_eval.generate_captions(val_loader, max_new_tokens=40)

bleu1 = cap_eval.bleu1_approx(cap_records)
print(f"Approximate BLEU-1: {bleu1:.4f}")

# Show a few examples
print("\nSample predictions:")
for rec in cap_records[:5]:
    print(f"  GT  : {rec['ground_truth']}")
    print(f"  Pred: {rec['prediction']}")
    print()

---
## Phase 5 · Ablation Studies & Interpretability

In [ ]:
# ── 5.1  Ablation study: compare adapter architectures ───────────────────────
# This cell trains and evaluates all three adapter types and logs results.
# Each run takes the same time as the baseline. Set max_train_samples small
# for a quick comparison.

from utils.config import get_config
from models.adapter import VisionLanguageModel

ABLATION_CONFIGS = ["baseline", "ablation_mlp", "ablation_perceiver", "ablation_contrastive"]
ablation_results = {}

for config_name in ABLATION_CONFIGS:
    print(f"\n{'='*60}")
    print(f"Running ablation: {config_name}")
    print(f"{'='*60}")

    ab_cfg = get_config(config_name)
    ab_cfg.coco_root      = cfg.coco_root
    ab_cfg.max_train_samples = 1000   # small for comparison speed
    ab_cfg.max_val_samples   = 200
    ab_cfg.num_epochs        = 1
    ab_cfg.output_dir        = f"/content/checkpoints/{config_name}"

    # Adapter kwargs
    kw = {}
    if ab_cfg.adapter_type == "mlp":
        kw["hidden_scale"] = ab_cfg.mlp_hidden_scale
    elif ab_cfg.adapter_type == "perceiver":
        kw["num_queries"] = ab_cfg.perceiver_num_queries
        kw["num_heads"]   = ab_cfg.perceiver_num_heads
        kw["num_layers"]  = ab_cfg.perceiver_num_layers

    ab_model = VisionLanguageModel(
        vision_model_name = ab_cfg.vision_model_name,
        lm_model_name     = ab_cfg.lm_model_name,
        adapter_type      = ab_cfg.adapter_type,
        use_cls_only      = ab_cfg.use_cls_only,
        vision_dim        = ab_cfg.vision_dim,
        text_dim          = ab_cfg.text_dim,
        **kw,
    )
    ab_model.adapter = ab_model.adapter.to(device)

    # Datasets (reuse loaders if config matches)
    ab_train_ds = COCOCaptionsDataset(ab_cfg.coco_root, "train", tokenizer, processor,
                                      max_samples=ab_cfg.max_train_samples)
    ab_val_ds   = COCOCaptionsDataset(ab_cfg.coco_root, "val",   tokenizer, processor,
                                      max_samples=ab_cfg.max_val_samples)
    ab_train_dl = build_dataloader(ab_train_ds, batch_size=16)
    ab_val_dl   = build_dataloader(ab_val_ds,   batch_size=16, shuffle=False)

    ab_trainer = Trainer(
        model        = ab_model,
        train_loader = ab_train_dl,
        val_loader   = ab_val_dl,
        output_dir   = ab_cfg.output_dir,
        num_epochs   = ab_cfg.num_epochs,
        learning_rate= ab_cfg.learning_rate,
        grad_accum_steps = ab_cfg.grad_accum_steps,
        use_contrastive  = ab_cfg.use_contrastive,
        log_every    = 20,
        save_every   = 999999,   # skip intermediate saves
    )
    ab_trainer.train()

    # Measure CKA after training
    cka = analyzer.compute_cka(ab_model, ab_val_dl, device=device, max_batches=10)
    ablation_results[config_name] = {
        "adapter_type":    ab_cfg.adapter_type,
        "num_params":      ab_model.num_trainable_params(),
        "cka":             cka,
    }
    del ab_model  # free VRAM
    torch.cuda.empty_cache()

print("\n" + "="*60)
print("ABLATION RESULTS SUMMARY")
print("="*60)
for name, res in ablation_results.items():
    print(f"{name:30s}  params={res['num_params']:>10,}  CKA={res['cka']:.4f}")

In [ ]:
# ── 5.2  Plot ablation results ────────────────────────────────────────────────
import matplotlib.pyplot as plt

names  = list(ablation_results.keys())
ckas   = [ablation_results[n]["cka"] for n in names]
params = [ablation_results[n]["num_params"] / 1e6 for n in names]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(names, ckas, color="steelblue")
axes[0].axhline(0.7, color="red", linestyle="--", label="Target CKA=0.7")
axes[0].set_ylabel("CKA Similarity")
axes[0].set_title("Embedding Alignment (CKA) by Adapter")
axes[0].legend()
axes[0].tick_params(axis="x", rotation=20)

axes[1].bar(names, params, color="tomato")
axes[1].set_ylabel("Trainable Params (M)")
axes[1].set_title("Adapter Size by Architecture")
axes[1].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.savefig("/content/ablation_results.png", dpi=150)
plt.show()

In [ ]:
# ── 5.3  Failure mode analysis ───────────────────────────────────────────────
# Categorise incorrect VQA answers by question type.

if 'vqa_results' in locals() and vqa_results.get("results"):
    from collections import defaultdict
    type_stats = defaultdict(lambda: {"correct": 0, "total": 0})

    for r in vqa_results["results"]:
        # Crude question type heuristics
        q = r.get("question", "").lower()
        if q.startswith("how many"):
            qtype = "counting"
        elif q.startswith("what color") or q.startswith("what is the color"):
            qtype = "color"
        elif q.startswith("is there") or q.startswith("are there"):
            qtype = "presence"
        elif q.startswith("where"):
            qtype = "spatial"
        else:
            qtype = "other"
        type_stats[qtype]["total"] += 1
        type_stats[qtype]["correct"] += r["correct"]

    print("VQA Accuracy by Question Type:")
    for qtype, s in sorted(type_stats.items()):
        acc = s["correct"] / max(1, s["total"])
        print(f"  {qtype:12s}: {acc:.3f}  ({s['correct']}/{s['total']})")
else:
    print("VQA results not available. Run Phase 4 cell 4.5 first.")

In [ ]:
# ── 5.4  Save final results summary ──────────────────────────────────────────
import json

summary = {
    "experiment":    cfg.experiment_name,
    "adapter_type":  cfg.adapter_type,
    "use_cls_only":  cfg.use_cls_only,
    "num_epochs":    cfg.num_epochs,
    "cka_before":    cka_before,
    "cka_after":     cka_after,
    "bleu1":         bleu1 if 'bleu1' in locals() else None,
    "vqa_accuracy":  vqa_results.get("accuracy") if 'vqa_results' in locals() else None,
    "ablation":      ablation_results if 'ablation_results' in locals() else {},
}

summary_path = "/content/results_summary.json"
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

print(f"Results saved → {summary_path}")
print(json.dumps(summary, indent=2))

---
## Appendix · Useful One-liners

In [ ]:
# ── A1  Check GPU memory usage ───────────────────────────────────────────────
if device == "cuda":
    alloc = torch.cuda.memory_allocated() / 1e9
    reserved = torch.cuda.memory_reserved() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"Allocated : {alloc:.2f} GB")
    print(f"Reserved  : {reserved:.2f} GB")
    print(f"Total     : {total:.2f} GB")

In [ ]:
# ── A2  Quick inference with your own image ───────────────────────────────────
from google.colab import files   # Colab-specific; remove if running locally
from io import BytesIO

uploaded = files.upload()  # upload an image from your machine
for fname, content in uploaded.items():
    img = Image.open(BytesIO(content)).convert("RGB")
    pv  = processor(images=img, return_tensors="pt")["pixel_values"].to(device)

    prompt_str = "Describe this image:"
    p_ids = tokenizer(prompt_str, return_tensors="pt")["input_ids"].to(device)

    model.eval()
    with torch.no_grad():
        gen = model.generate(pv, p_ids, torch.ones_like(p_ids), max_new_tokens=60)
    print(tokenizer.decode(gen[0][p_ids.shape[-1]:], skip_special_tokens=True))

In [ ]:
# ── A3  Export adapter weights for sharing ────────────────────────────────────
export_path = "/content/adapter_final_weights.pt"
torch.save(model.adapter.state_dict(), export_path)
print(f"Adapter weights saved → {export_path}")
print(f"File size: {os.path.getsize(export_path) / 1e6:.1f} MB")